# EEE3032 – Stage 7: Extract Frozen MobileNetV3-Small Descriptors

This notebook begins the optional **pretrained deep descriptors (+10)** extension.

**Question behind the extension:** can a frozen ImageNet-pretrained MobileNetV3-Small descriptor provide information that complements the selected SIFT-BoVW representation, and does late score fusion improve retrieval on the MSRC-v2 development queries?

In this stage we:

1. load the same official gallery and development split used in Stages 5–6;
2. use the supplied, frozen MobileNetV3-Small extractor (576 values per image);
3. L2-normalise each feature vector so that dot products represent cosine similarity;
4. cache gallery and development descriptors separately, in manifest order.

The network is **not trained or fine-tuned**. The final test queries are not passed through the network in this stage. Do not add test images to either cache.

Run this notebook from top to bottom. The first run may download the official pretrained ImageNet weights.

## 1. Connect to the project and set paths

Use the same `PROJECT_ROOT` and `DATASET_SOURCE` values as in Stage 6. `PROJECT_ROOT` must contain `cvpr/`, `split_manifest.csv`, and the saved Stage 5 feature folders. If the project folder does not contain `cvpr/`, copy that folder from the supplied coursework starter ZIP into the project root.

In [1]:
from pathlib import Path
import sys
import csv
import numpy as np

# Mount Google Drive when running in Colab. This cell is harmless outside Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/cvpr_coursework/EEE3032_Coursework_Starter_v3"
)
DATASET_SOURCE = Path(
    "/content/drive/MyDrive/cvpr_coursework/MSRC_ObjCategImageDatabase_v2"
)

if not (PROJECT_ROOT / "cvpr" / "deep.py").is_file():
    raise FileNotFoundError(
        "cvpr/deep.py was not found. Set PROJECT_ROOT to the coursework project "
        "root and make sure the cvpr folder from the supplied starter ZIP is present."
    )
if not (PROJECT_ROOT / "split_manifest.csv").is_file():
    raise FileNotFoundError(f"split_manifest.csv was not found in {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Dataset source:", DATASET_SOURCE)

Mounted at /content/drive
Project: /content/drive/MyDrive/cvpr_coursework/EEE3032_Coursework_Starter_v3
Dataset source: /content/drive/MyDrive/cvpr_coursework/MSRC_ObjCategImageDatabase_v2


## 2. Import the supplied frozen extractor and load the official split

The official split loader returns the test records as part of its interface. We deliberately do not compute, cache, inspect, or evaluate any test descriptors here.

In [ ]:
from cvpr.dataset import load_split_records
from cvpr.retrieval import compute_descriptors
from cvpr.deep import (
    make_mobilenet_v3_small_extractor,
    save_feature_cache,
)

MANIFEST = PROJECT_ROOT / "split_manifest.csv"

gallery, dev, _test_records = load_split_records(
    DATASET_SOURCE,
    manifest_path=MANIFEST,
    extract_dir=PROJECT_ROOT / "data",
)

assert len(gallery) == 391, len(gallery)
assert len(dev) == 100, len(dev)

# We use only these two lists from this point onwards.
print(f"Gallery images: {len(gallery)}")
print(f"Development queries: {len(dev)}")
print("Test images: not used in Stage 7")

Gallery images: 391
Development queries: 100
Test images: not used in Stage 7


## 3. Set the model and cache locations

MobileNetV3-Small returns a 576-dimensional pooled feature vector. The official torchvision preprocessing is applied inside the supplied extractor. Setting `l2_normalize=True` normalises each descriptor before it is saved.

In [ ]:
MODEL_NAME = "mobilenet_v3_small"
DESCRIPTOR_NAME = "mobilenet_v3_small_l2"
FEATURE_DIM = 576
L2_NORMALIZE = True

DEEP_CACHE_DIR = PROJECT_ROOT / "deep_features" / "stage7"
DEEP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

GALLERY_CACHE_FILE = DEEP_CACHE_DIR / "mobilenet_v3_small_gallery.npz"
DEV_CACHE_FILE = DEEP_CACHE_DIR / "mobilenet_v3_small_dev.npz"

print("Model:", MODEL_NAME)
print("Descriptor dimension:", FEATURE_DIM)
print("L2-normalised:", L2_NORMALIZE)
print("Cache folder:", DEEP_CACHE_DIR)

Model: mobilenet_v3_small
Descriptor dimension: 576
L2-normalised: True
Cache folder: /content/drive/MyDrive/cvpr_coursework/EEE3032_Coursework_Starter_v3/deep_features/stage7


## 4. Reuse a valid cache or extract descriptors

The cache loader checks the descriptor name, dimension, numerical values and filenames. It reorders rows if the saved filename order differs, and refuses to continue if any requested image is missing. This avoids silently pairing a feature vector with the wrong image.

In [ ]:
def load_valid_deep_cache(cache_file, records, expected_dim, descriptor_name):
    """Load a compatible cache and return rows in the requested record order."""
    if not cache_file.exists():
        return None

    try:
        with np.load(cache_file, allow_pickle=False) as saved:
            saved_names = saved["filenames"].astype(str).tolist()
            saved_features = np.asarray(saved["features"], dtype=np.float32)
            saved_descriptor_name = str(saved["descriptor_name"].item())
    except Exception as exc:
        print(f"Cache cannot be read ({cache_file.name}): {exc}")
        return None

    expected_names = [record.filename for record in records]
    if saved_descriptor_name != descriptor_name:
        print(f"Cache descriptor metadata mismatch: {cache_file.name}")
        return None
    if saved_features.ndim != 2 or saved_features.shape[1] != expected_dim:
        print(f"Cache dimension mismatch: {cache_file.name} {saved_features.shape}")
        return None
    if len(saved_names) != len(saved_features) or len(set(saved_names)) != len(saved_names):
        print(f"Cache filename metadata is invalid: {cache_file.name}")
        return None
    if not np.isfinite(saved_features).all():
        print(f"Cache contains non-finite values: {cache_file.name}")
        return None

    lookup = {name: saved_features[i] for i, name in enumerate(saved_names)}
    missing = [name for name in expected_names if name not in lookup]
    if missing:
        print(f"Cache is missing {len(missing)} images: {missing[:3]}")
        return None

    return np.vstack([lookup[name] for name in expected_names]).astype(np.float32)


gallery_deep = load_valid_deep_cache(
    GALLERY_CACHE_FILE, gallery, FEATURE_DIM, DESCRIPTOR_NAME
)
dev_deep = load_valid_deep_cache(
    DEV_CACHE_FILE, dev, FEATURE_DIM, DESCRIPTOR_NAME
)

need_gallery = gallery_deep is None
need_dev = dev_deep is None

if need_gallery or need_dev:
    print("Loading the frozen pretrained MobileNetV3-Small model...")
    extractor = make_mobilenet_v3_small_extractor(l2_normalize=L2_NORMALIZE)

    if need_gallery:
        print("Extracting gallery descriptors...")
        gallery_deep = compute_descriptors(gallery, extractor)
        save_feature_cache(
            GALLERY_CACHE_FILE,
            gallery,
            gallery_deep,
            descriptor_name=DESCRIPTOR_NAME,
        )
        print("Saved:", GALLERY_CACHE_FILE)

    if need_dev:
        print("Extracting development-query descriptors...")
        dev_deep = compute_descriptors(dev, extractor)
        save_feature_cache(
            DEV_CACHE_FILE,
            dev,
            dev_deep,
            descriptor_name=DESCRIPTOR_NAME,
        )
        print("Saved:", DEV_CACHE_FILE)
else:
    print("Both valid descriptor caches already exist; reusing them.")

assert gallery_deep.shape == (391, FEATURE_DIM), gallery_deep.shape
assert dev_deep.shape == (100, FEATURE_DIM), dev_deep.shape
assert np.isfinite(gallery_deep).all()
assert np.isfinite(dev_deep).all()

## 5. Validate descriptor norms and save a summary

Because the descriptors are L2-normalised, every non-zero row should have a norm close to 1. This is important because Stage 8 combines cosine-similarity scores from the deep representation and the L2-normalised BoVW representation.

In [ ]:
gallery_norms = np.linalg.norm(gallery_deep, axis=1)
dev_norms = np.linalg.norm(dev_deep, axis=1)

assert np.all(gallery_norms > 0), "A gallery descriptor is zero."
assert np.all(dev_norms > 0), "A development descriptor is zero."
assert np.allclose(gallery_norms, 1.0, atol=1e-4)
assert np.allclose(dev_norms, 1.0, atol=1e-4)

summary_rows = [
    {
        "split": "gallery",
        "n_images": len(gallery),
        "feature_dimension": gallery_deep.shape[1],
        "minimum_l2_norm": float(gallery_norms.min()),
        "mean_l2_norm": float(gallery_norms.mean()),
        "maximum_l2_norm": float(gallery_norms.max()),
        "cache_file": str(GALLERY_CACHE_FILE),
    },
    {
        "split": "development",
        "n_images": len(dev),
        "feature_dimension": dev_deep.shape[1],
        "minimum_l2_norm": float(dev_norms.min()),
        "mean_l2_norm": float(dev_norms.mean()),
        "maximum_l2_norm": float(dev_norms.max()),
        "cache_file": str(DEV_CACHE_FILE),
    },
]

summary_file = DEEP_CACHE_DIR / "stage7_deep_descriptor_summary.csv"
with summary_file.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

print("Gallery descriptor matrix:", gallery_deep.shape)
print("Development descriptor matrix:", dev_deep.shape)
print("Gallery norm range:", (float(gallery_norms.min()), float(gallery_norms.max())))
print("Development norm range:", (float(dev_norms.min()), float(dev_norms.max())))
print("Saved summary:", summary_file)
print("\nStage 7 complete. No final test descriptors were extracted or evaluated.")

## What Stage 8 will use

Stage 8 loads these two cached matrices and the selected Stage 5 BoVW matrices. It compares:

- **Deep-only:** MobileNetV3-Small cosine similarity;
- **BoVW-only:** the selected SIFT-BoVW cosine similarity;
- **Late fusion:** `S_fused = α × S_BoVW + (1 − α) × S_deep`.

The fusion weight `α` is selected using development mAP (with mean P@10 as a tie-breaker). The chosen value is then frozen before any final test evaluation.